# DeepWeeds Lab Day 2 — executable workflow
Notebook điều phối bản cài đặt trong `code/`. Mọi lựa chọn dùng validation; test chỉ bật ở ô chung kết.

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/duongduong1606/K4-Track4-Day2-Deeplearning-Advance.git'
if importlib.util.find_spec('google.colab') is not None:
    repo_dir = Path('/content/K4-Track4-Day2-Deeplearning-Advance')
    if not repo_dir.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(repo_dir)])
    else:
        # A Colab runtime may keep an older /content clone across notebook reopenings.
        # Fast-forward it so all committed dataset/checkpoint fixes are always loaded.
        subprocess.check_call(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'])
    os.chdir(repo_dir)
    commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
    print(f'Running repository commit: {commit}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'code/requirements.txt'])

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import torch
sys.path.insert(0, str(Path('code').resolve()))
import dataset, inference, benchmark, diagnostics, reporting, colab_automation, final_inference, post_run
from train import Config, run
from experiment_plan import BACKBONES, TRAINING, COMBINED_CANDIDATE, final_configs
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
IN_COLAB = importlib.util.find_spec('google.colab') is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/DeepWeedsLab')
    LOCAL_ROOT = Path('/content/deepweeds')
    staged = colab_automation.stage_dataset(DRIVE_ROOT, LOCAL_ROOT)
    IMAGES_DIR, LABELS_DIR = Path(staged['images_dir']), Path(staged['labels_dir'])
else:
    DRIVE_ROOT = Path('.')
    LOCAL_ROOT = Path('.')
    IMAGES_DIR, LABELS_DIR = Path('data/images'), Path('data/labels')
print(IMAGES_DIR, LABELS_DIR)

## 1. Kiểm tra dữ liệu và split

In [ ]:
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_report = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
display(pd.DataFrame(split_report['per_class']).rename_axis('Label'))
display(pd.Series(split_report['n'], name='n_images'))

## 2. Unit tests trước khi huấn luyện

In [ ]:
!python -m unittest code.test_implementation -v
!python -m unittest discover -s tests -v

## 3. So sánh backbone
Đặt cờ thành `True` khi đã xem ảnh augmentation và kiểm tra overfit một batch.

In [ ]:
RUN_PIPELINE_CHECKS = False
check_cfg = Config(images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR))
pipeline_report = diagnostics.run_pipeline_checks(check_cfg, steps=200) if RUN_PIPELINE_CHECKS else None
pipeline_report

In [ ]:
SCREENING_MANIFEST = DRIVE_ROOT / 'queue/screening.json'
colab_automation.create_screening_manifest(SCREENING_MANIFEST)
RUN_SCREENING_QUEUE = False
screening_results = (
    colab_automation.run_queue(SCREENING_MANIFEST, DRIVE_ROOT, LOCAL_ROOT, max_runs=3)
    if RUN_SCREENING_QUEUE else []
)
pd.DataFrame(screening_results)

## 4. Ablation công thức huấn luyện
Mỗi `T01…T09` chỉ khác `T00` một yếu tố. Sửa backbone đồng loạt sau khi chọn bằng kết quả validation.

In [ ]:
screening_manifest = json.loads(SCREENING_MANIFEST.read_text(encoding='utf-8'))
pd.DataFrame([{**{'run_id': r['run_id'], 'status': r['status'], 'attempts': r['attempts']}, **r['config']} for r in screening_manifest['runs']])

In [ ]:
# Chỉ thêm COMBINED_CANDIDATE vào manifest sau khi các thành phần thắng rõ trên validation.
COMBINED_CANDIDATE

## 5. Suy luận trên validation
Nạp checkpoint đã chọn, sau đó dùng `predict_logits`, `aggregate_views`, `fit_temperature`, `latency_report`. Không dùng test để chọn phương pháp.

In [ ]:
SELECTED_RUN_ROOT = DRIVE_ROOT / 'artifacts/runs/B01/seed42'  # thay bằng run thắng trên val
INFERENCE_OUTPUT = DRIVE_ROOT / 'artifacts/runs'
RUN_INFERENCE_SWEEP = False
inference_table = (
    final_inference.run_validation_sweep(
        SELECTED_RUN_ROOT, IMAGES_DIR, LABELS_DIR, INFERENCE_OUTPUT, latency_iters=100
    ) if RUN_INFERENCE_SWEEP else pd.DataFrame()
)
display(inference_table)

In [ ]:
FINAL_SPEC = DRIVE_ROOT / 'queue/final_inference_spec.json'
FREEZE_INFERENCE_SPEC = False
if FREEZE_INFERENCE_SPEC:
    final_inference.freeze_inference_spec(
        INFERENCE_OUTPUT / 'inference_results.csv',
        method='I09', model_exp_id='B01', seed=42, output_path=FINAL_SPEC,
        selection_note='I09 được chọn theo macro-F1, ECE và p95 trên validation',
    )
FINAL_SPEC

## 6. Chung kết
Điền cấu hình thắng trên validation. Lần chạy đầu để `ENABLE_FINAL_TEST=False`; chỉ bật một lần sau khi đã chốt mọi lựa chọn.

In [ ]:
BEST_ON_VAL = Config(exp_id='F01', backbone='resnet50')  # thay bằng cấu hình thực sự thắng trên val
FINAL_TRAIN_MANIFEST = DRIVE_ROOT / 'queue/final_training.json'
CREATE_FINAL_TRAINING = False
if CREATE_FINAL_TRAINING:
    colab_automation.create_final_training_manifest(
        FINAL_TRAIN_MANIFEST, BEST_ON_VAL,
        selection_note='F01 được chốt hoàn toàn từ kết quả validation',
    )
RUN_FINAL_TRAINING = False
final_results = (
    colab_automation.run_queue(FINAL_TRAIN_MANIFEST, DRIVE_ROOT, LOCAL_ROOT, max_runs=1)
    if RUN_FINAL_TRAINING else []
)
pd.DataFrame(final_results)

In [ ]:
FINAL_TEST_MANIFEST = DRIVE_ROOT / 'queue/final_test.json'
CREATE_FINAL_TEST = False
if CREATE_FINAL_TEST:
    colab_automation.create_final_test_manifest(
        FINAL_TEST_MANIFEST, DRIVE_ROOT, FINAL_SPEC
    )
# Chỉ mở khóa sau khi final training đủ 6 run và spec đã đóng băng:
# colab_automation.unlock_final_manifest(FINAL_TEST_MANIFEST, 'Đã chốt recipe và inference từ validation')
RUN_ONE_FINAL_TEST = False
test_results = (
    colab_automation.run_final_test_queue(
        FINAL_TEST_MANIFEST, DRIVE_ROOT, LOCAL_ROOT, max_runs=1, allow_test=True
    ) if RUN_ONE_FINAL_TEST else []
)
pd.DataFrame(test_results)

## 7. Kiểm tra kết quả cuối
```bash
python eval.py score --pred "predictions/F01_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out
python eval.py grade --final "predictions/F01_seed*_test.csv" --baseline "predictions/T00_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv
```

In [ ]:
xlsx_path = reporting.build_results_workbook(
    DRIVE_ROOT / 'results.xlsx',
    runs_dir=DRIVE_ROOT / 'artifacts/runs',
    pred_dir=DRIVE_ROOT / 'artifacts/predictions',
)
xlsx_path

In [ ]:
RUN_POSTPROCESS = False
STUDENT_ID = 'MSSV'
STUDENT_NAME = 'ho_ten_khong_dau'
NOTEBOOK_URL = 'https://colab.research.google.com/...'
submission_dir = DRIVE_ROOT / 'submissions' / f'{STUDENT_ID}_{STUDENT_NAME}'
submission = (
    post_run.assemble_submission(
        DRIVE_ROOT, FINAL_TEST_MANIFEST, Path.cwd(), submission_dir, NOTEBOOK_URL
    ) if RUN_POSTPROCESS else None
)
submission